In [1]:
import pandas as pd

df = pd.read_csv("dataset/synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,27/06/2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,12/07/2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,02/06/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [2]:
df.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI', 'LegacyCRM'], dtype=object)

In [3]:
df.target_label.unique()

array(['HTTP Status', 'Critical Error', 'Security Alert', 'Error',
       'System Notification', 'Resource Usage', 'User Action',
       'Workflow Error', 'Deprecation Warning'], dtype=object)

In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN


import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Generating embeddings:

embeddings = model.encode(df['log_message'].tolist())
embeddings[:3]


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[-0.10293972,  0.03354599, -0.02202604, ...,  0.00457789,
        -0.04259715,  0.00322623],
       [ 0.00804573, -0.03573925,  0.04938737, ...,  0.01538317,
        -0.06230951, -0.02774668],
       [-0.00908224,  0.13003926, -0.05275577, ...,  0.02014109,
        -0.05117095, -0.02930295]], dtype=float32)

In [5]:
dbscan = DBSCAN(eps = 0.15, min_samples = 1, metric = 'cosine')
clusters = dbscan.fit_predict(embeddings)

df['cluster'] = clusters
df.head()

,timestamp,source,log_message,target_label,complexity,cluster
0,27/06/2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2
3,12/07/2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0
4,02/06/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0


In [6]:
df[df.cluster == 1]

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
248,05/02/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1
361,11/19/2025 23:06,BillingSystem,Email service experienced a sending issue,Error,bert,1
450,10/27/2025 5:59,ThirdPartyAPI,Email delivery system encountered an error,Error,bert,1
477,12/02/2025 10:30,AnalyticsEngine,Email transmission error caused service impact,Critical Error,bert,1
570,11/07/2025 18:08,ThirdPartyAPI,Email service impacted by sending failure,Critical Error,bert,1
678,4/28/2025 15:13,AnalyticsEngine,Email delivery problem affected system,Critical Error,bert,1
695,6/30/2025 12:46,ThirdPartyAPI,Service outage due to email delivery problem,Critical Error,bert,1
816,07/09/2025 09:38,BillingSystem,Error in email delivery affected service,Critical Error,bert,1


In [7]:
cluster_counts = df['cluster'].value_counts()
large_clusters = cluster_counts[cluster_counts > 10].index

for cluster in large_clusters:
    print(f"Cluster {cluster}:")
    print(df[df['cluster'] == cluster]['log_message'].head(5).to_string(index = False))
    print()

Cluster 0:
nova.osapi_compute.wsgi.server [req-b9718cd8-f6...
nova.osapi_compute.wsgi.server [req-4895c258-b2...
nova.osapi_compute.wsgi.server [req-ee8bc8ba-92...
nova.osapi_compute.wsgi.server [req-f0bffbc3-5a...
nova.osapi_compute.wsgi.server [req-2bf7cfee-a2...

Cluster 5:
nova.compute.claims [req-a07ac654-8e81-416d-bfb...
nova.compute.claims [req-d6986b54-3735-4a42-907...
nova.compute.claims [req-72b4858f-049e-49e1-b31...
nova.compute.claims [req-5c8f52bd-8e3c-41f0-95a...
nova.compute.claims [req-d38f479d-9bb9-4276-968...

Cluster 12:
User User685 logged out.
 User User395 logged in.
 User User225 logged in.
User User494 logged out.
 User User900 logged in.

Cluster 14:
Backup started at 2025-05-14 07:06:55.
Backup started at 2025-02-15 20:00:19.
  Backup ended at 2025-08-08 13:06:23.
Backup started at 2025-11-14 08:27:43.
Backup started at 2025-12-09 10:19:11.

Cluster 9:
Backup completed successfully.
Backup completed successfully.
Backup completed successfully.
Backup completed

In [8]:
import re
def classify_with_regex(log_message):
    regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }

    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE): 
            return label
    return None

In [9]:
classify_with_regex("Account with ID AIO2938481 created by Boss")

'User Action'

In [10]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)
df[df.regex_label.notnull()]

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
7,10/11/2025 08:44,ModernHR,File data_6169.csv uploaded successfully by us...,System Notification,regex,4,System Notification
14,01/04/2025 01:43,ThirdPartyAPI,File data_3847.csv uploaded successfully by us...,System Notification,regex,4,System Notification
15,05/01/2025 09:41,ModernCRM,Backup completed successfully.,System Notification,regex,9,System Notification
18,2/22/2025 17:49,ModernCRM,Account with ID 5351 created by User634.,User Action,regex,10,User Action
27,9/24/2025 19:57,ThirdPartyAPI,User User685 logged out.,User Action,regex,12,User Action
...,...,...,...,...,...,...,...
2376,6/27/2025 8:47,ModernCRM,System updated to version 2.0.5.,System Notification,regex,22,System Notification
2381,09/05/2025 06:39,ThirdPartyAPI,Disk cleanup completed successfully.,System Notification,regex,36,System Notification
2394,04/03/2025 13:13,ModernHR,Disk cleanup completed successfully.,System Notification,regex,36,System Notification
2395,05/02/2025 14:29,ThirdPartyAPI,Backup ended at 2025-05-06 11:23:16.,System Notification,regex,14,System Notification


In [11]:
df.shape

(2410, 7)

In [12]:
df_non_regex = df[df['regex_label'].isnull()].copy()
df_non_regex.shape

(1910, 7)

In [13]:
df_non_regex

,timestamp,source,log_message,target_label,complexity,cluster,regex_label
0,27/06/2025 07:20,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert,0,None
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1,None
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert,2,None
3,12/07/2025 00:24,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert,0,None
4,02/06/2025 18:25,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert,0,None
...,...,...,...,...,...,...,...
2405,13/08/2025 07:29,ModernHR,nova.osapi_compute.wsgi.server [req-96c3ec98-2...,HTTP Status,bert,0,None
2406,01/11/2025 05:32,ModernHR,User 3844 account experienced multiple failed ...,Security Alert,bert,8,None
2407,03/08/2025 03:07,ThirdPartyAPI,nova.metadata.wsgi.server [req-b6d4a270-accb-4...,HTTP Status,bert,0,None
2408,11/11/2025 11:52,BillingSystem,Email service affected by failed transmission,Critical Error,bert,1,None


In [14]:
print(df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() <= 10].index.tolist())

['Workflow Error', 'Deprecation Warning']


In [15]:
df_non_legacy = df_non_regex[df_non_regex.source != 'LegacyCRM']
df_non_legacy.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [16]:
filtered_embeddings = model.encode(df['log_message'].tolist())
filtered_embeddings[:3]


array([[-0.10293972,  0.03354599, -0.02202604, ...,  0.00457789,
        -0.04259715,  0.00322623],
       [ 0.00804573, -0.03573925,  0.04938737, ...,  0.01538317,
        -0.06230951, -0.02774668],
       [-0.00908224,  0.13003926, -0.05275577, ...,  0.02014109,
        -0.05117095, -0.02930295]], dtype=float32)

In [17]:
print("X shape:", len(X))
print("y shape:", len(y))

NameError: name 'X' is not defined

In [ ]:
print(df_non_legacy.columns)

Index(['timestamp', 'source', 'log_message', 'target_label', 'complexity',
       'cluster', 'regex_label'],
      dtype='object')


In [ ]:
df = df_non_legacy.dropna(subset = ['log_message', 'target_label'])

X = model.encode(df['log_message'].tolist())
y = df['target_label']

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = model.encode(df['log_message'].tolist())
y = df['target_label']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 45)
clf = LogisticRegression(max_iter = 1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

                precision    recall  f1-score   support

Critical Error       1.00      0.97      0.99        37
         Error       1.00      1.00      1.00        38
   HTTP Status       1.00      1.00      1.00       211
Resource Usage       1.00      1.00      1.00        35
Security Alert       0.98      1.00      0.99        60

      accuracy                           1.00       381
     macro avg       1.00      0.99      1.00       381
  weighted avg       1.00      1.00      1.00       381



In [ ]:
import joblib

joblib.dump(clf, '../models/log_classifier.joblib')

['../models/log_classifier.joblib']

(2410, 384)
(1903,)
